In [1]:
%pip install statsbombpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from statsbombpy import sb

In [3]:
import glob
import json
import os

In [4]:
# Path to the cloned 'data' folder
DATA_DIR = "./open-data/data"

In [5]:
# 1. Load Competitions & Matches
with open(os.path.join(DATA_DIR, "competitions.json"), "r", encoding="utf-8") as f:
    competitions = json.load(f)

match_files = glob.glob(os.path.join(DATA_DIR, "matches", "*", "*.json"))
matches_list = []
for file in match_files:
    with open(file, "r", encoding="utf-8") as f:
        matches_list.extend(json.load(f))

matches_df = pd.json_normalize(matches_list).drop_duplicates(subset=["match_id"])

In [6]:
print(f"Total Matches: {len(matches_df)}")

Total Matches: 3961


In [7]:
unique_teams = pd.unique(
    matches_df[["home_team.home_team_name", "away_team.away_team_name"]].values.ravel("K")
)
print(f"Total Teams: {len(unique_teams)}")

Total Teams: 353


In [8]:
# 2. Extract Penalties from Event Files
event_files = glob.glob(os.path.join(DATA_DIR, "events", "*.json"))
penalties = []

for file in event_files:
    with open(file, "r", encoding="utf-8") as f:
        events = json.load(f)

    # First pass: map goalkeepers per team for this match to identify facing keepers
    team_keepers = {}
    for ev in events:
        if ev.get("type", {}).get("name") == "Starting XI":
            team_name = ev.get("team", {}).get("name")
            for player in ev.get("tactics", {}).get("lineup", []):
                if player.get("position", {}).get("name") == "Goalkeeper":
                    team_keepers[team_name] = player.get("player", {}).get("name")

    # Second pass: find penalty shots
    for ev in events:
        if (
            ev.get("type", {}).get("name") == "Shot"
            and ev.get("shot", {}).get("type", {}).get("name") == "Penalty"
        ):
            shooting_team = ev.get("team", {}).get("name")
            
            # Find the opposing keeper from the freeze frame or starting XI map
            keeper_name = None
            freeze_frame = ev.get("shot", {}).get("freeze_frame", [])
            for player_frame in freeze_frame:
                if player_frame.get("position", {}).get("name") == "Goalkeeper":
                    keeper_name = player_frame.get("player", {}).get("name")
                    break

            # Fallback if keeper wasn't tagged in the freeze frame
            if not keeper_name:
                for team, gk in team_keepers.items():
                    if team != shooting_team:
                        keeper_name = gk
                        break

            penalties.append(
                {
                    "match_id": ev.get("match_id"),
                    "period": ev.get("period"),
                    "minute": ev.get("minute"),
                    "player": ev.get("player", {}).get("name"),
                    "team": shooting_team,
                    "keeper": keeper_name,
                    "outcome": ev.get("shot", {}).get("outcome", {}).get("name"),
                }
            )

penalties_df = pd.DataFrame(penalties)

# Answer Q3: Total Penalties
print(f"Total Penalties: {len(penalties_df)}")

Total Penalties: 1557


In [9]:
penalties_df["period"].unique()

inside_regular = penalties_df[penalties_df["period"].isin([1, 2])]
print(f"Penalties inside regular time: {len(inside_regular)}")

Penalties inside regular time: 1186


In [10]:
# Answer Q4: Penalties not during regular time (Period > 2)
outside_regular = penalties_df[~penalties_df["period"].isin([1, 2])]
print(f"Penalties outside regular time: {len(outside_regular)}")
print(outside_regular["period"].value_counts().rename(index={3: "Extra Time 1", 4: "Extra Time 2", 5: "Shootout"}))

Penalties outside regular time: 371
period
Shootout        363
Extra Time 1      4
Extra Time 2      4
Name: count, dtype: int64


In [11]:
# Answer Q5: Penalties per Player
print("\nTop 10 Penalty Takers:")
print(penalties_df["player"].value_counts().head(50))


Top 10 Penalty Takers:
player
Lionel Andrés Messi Cuccittini         83
Cristiano Ronaldo dos Santos Aveiro    23
Ronaldo de Assis Moreira               16
Neymar da Silva Santos Junior          15
Harry Kane                             14
Kylian Mbappé Lottin                   12
Kim Little                             10
Luis Alberto Suárez Díaz               10
Antoine Griezmann                       9
Moritz Hartmann                         8
Chloe Kelly                             8
Antonio Candreva                        8
Fara Williams                           8
Josip Iličić                            8
María Francesca Caldentey Oliver        7
Paulo Bruno Exequiel Dybala             7
Nikita Parris                           7
Zlatan Ibrahimović                      7
Megan Anna Rapinoe                      7
Thierry Henry                           7
Troy Deeney                             7
Youssef El-Arabi                        7
Luka Modrić                             6
Jam

In [12]:
# Answer Q6: Penalties per Keeper
print("\nTop 10 Keepers Facing Penalties:")
print(penalties_df["keeper"].value_counts().head(50))


Top 10 Keepers Facing Penalties:
keeper
Víctor Valdés Arribas         29
Pauline Peyraud Magnin        25
Yann Sommer                   24
Gianluigi Donnarumma          20
Hugo Lloris                   20
Ronwen Williams               19
Jordan Pickford               18
Damián Emiliano Martínez      16
Hannah Hampton                16
Kasper Schmeichel             15
Alyssa Michele Naeher         14
Unai Simón Mendibil           14
Danijel Subašić               14
Ann-Katrin Berger             14
Mackenzie Arnold              14
Keylor Navas Gamboa           14
Mary Alexandra Earps          12
Marc-André ter Stegen         12
Megan Walsh                   12
Vicente Guaita Panadero       12
Jan Oblak                     11
Sophie Baggaley               11
Paul Bernardoni               11
Sergio Rochet Álvarez         11
Lionel Mpasi-Nzau             11
Igor Akinfeev                 11
Wojciech Szczęsny             10
Dominik Livaković             10
Mohamed Abougabal              9
Em

In [13]:
matches_df["match_date"] = pd.to_datetime(matches_df["match_date"])

# 3. Overall Time Range
earliest_match = matches_df["match_date"].min()
latest_match = matches_df["match_date"].max()
total_span_days = (latest_match - earliest_match).days

print("=" * 50)
print("OVERALL STATSBOMB OPEN DATA TIME RANGE")
print("=" * 50)
print(f"Earliest Match Date : {earliest_match.strftime('%Y-%m-%d')}")
print(f"Latest Match Date   : {latest_match.strftime('%Y-%m-%d')}")
print(f"Total Span          : {total_span_days} days (~{total_span_days // 365} years)")

OVERALL STATSBOMB OPEN DATA TIME RANGE
Earliest Match Date : 1958-06-24
Latest Match Date   : 2025-07-27
Total Span          : 24505 days (~67 years)


In [21]:
matches_df.sort_values(by="match_date", ascending=False).head(10)

,match_id,match_date,kick_off,home_score,away_score,match_status,match_status_360,last_updated,last_updated_360,match_week,...,competition_stage.id,competition_stage.name,stadium.id,stadium.name,stadium.country.id,stadium.country.name,referee.id,referee.name,referee.country.id,referee.country.name
3208,4020846,2025-07-27,16:00:00.000,1,1,available,available,2026-04-27T11:37:00.427234,2026-04-27T11:38:28.570776,6,...,26,Final,523.0,St. Jakob-Park,221.0,Switzerland,128.0,Stéphanie Frappart,78.0,France
3209,4020077,2025-07-23,19:00:00.000,0,1,available,available,2026-04-27T22:02:42.690507,2026-04-27T22:03:28.087062,5,...,15,Semi-finals,527.0,Stadion Letzigrund,221.0,Switzerland,2426.0,Edina Alves Batista,31.0,Brazil
3210,4020005,2025-07-22,19:00:00.000,2,1,available,available,2026-04-27T11:44:05.670023,2026-04-27T11:46:34.949735,5,...,15,Semi-finals,524.0,Stade de Genève,221.0,Switzerland,954.0,Ivana Martinčić,56.0,Croatia
3211,4018357,2025-07-19,19:00:00.000,1,1,available,available,2026-04-27T11:43:01.894527,2026-04-27T11:44:28.337054,4,...,11,Quarter-finals,523.0,St. Jakob-Park,221.0,Switzerland,830.0,Tess Olofsson,220.0,Sweden
3212,4018356,2025-07-18,19:00:00.000,2,0,available,available,2026-04-27T11:41:52.033839,2026-04-27T11:42:31.425560,4,...,11,Quarter-finals,115209.0,Stadion Wankdorf,221.0,Switzerland,1608.0,Maria Sole Ferrieri Caputi,112.0,Italy
3213,4018355,2025-07-17,19:00:00.000,2,2,available,available,2026-04-27T11:40:50.712285,2026-04-27T11:42:57.366830,4,...,11,Quarter-finals,527.0,Stadion Letzigrund,221.0,Switzerland,960.0,Marta Huerta de Aza,214.0,Spain
3214,4018354,2025-07-16,19:00:00.000,1,2,available,available,2026-04-27T11:39:48.518532,2026-04-27T11:40:31.980958,4,...,11,Quarter-finals,524.0,Stade de Genève,221.0,Switzerland,128.0,Stéphanie Frappart,78.0,France
3215,3998858,2025-07-13,19:00:00.000,6,1,available,available,2026-04-27T09:42:32.051183,2026-04-27T09:45:08.576665,3,...,10,Group Stage,4753.0,Kybunpark,221.0,Switzerland,956.0,Frida Mia Klarlund Nielsen,61.0,Denmark
3216,3998859,2025-07-13,19:00:00.000,2,5,available,available,2026-04-27T09:41:29.621678,2026-04-27T09:44:46.015760,3,...,10,Group Stage,523.0,St. Jakob-Park,221.0,Switzerland,954.0,Ivana Martinčić,56.0,Croatia
3217,3998857,2025-07-12,19:00:00.000,3,2,available,available,2026-04-27T09:50:04.593527,2026-04-27T09:51:48.882914,3,...,10,Group Stage,4441.0,swissporarena,221.0,Switzerland,2235.0,Olatz Rivera Olmedo,214.0,Spain
